In this notebook, I want to try out Sympy and maybe get close to calculating some jacobian matrices.

In [56]:
import sympy as sym
import numpy as np
import dill

In [34]:
sym.init_printing(use_unicode=True)

# State and Input
## State

In [60]:
q_w, q_x, q_y, q_z = sym.symbols("q_w, q_x, q_y, q_z")
v_x, v_y, v_z = sym.symbols("v_x, v_y, v_z")
p_x, p_y, p_z = sym.symbols("p_x, p_y, p_z")

x_q = sym.Matrix([q_w, q_x, q_y, q_z])
x_v = sym.Matrix([v_x, v_y, v_z])
x_p = sym.Matrix([p_x, p_y, p_z])

x = sym.Matrix.vstack(x_q, x_v, x_p)
x.T

[q_w  qₓ  q_y  q_z  vₓ  v_y  v_z  pₓ  p_y  p_z]

## Inputs

In [61]:
dt = sym.Symbol('\\Delta t')
gravity = sym.Symbol('g_{gravity}')

# ---

a_x, a_y, a_z = sym.symbols('a_x, a_y, a_z')
g_x, g_y, g_z = sym.symbols('g_x, g_y, g_z')

u_a = sym.Matrix([a_x, a_y, a_z])
u_g = sym.Matrix([g_x, g_y, g_z])

u = sym.Matrix.vstack(u_a, u_g)
u.T

[aₓ  a_y  a_z  gₓ  g_y  g_z]

# Helper functions

For Hamilton product, I turn the quaternion into a matrix to allow for matrix multiplication.
Inversion is trivial.
For quaternion normalization, I also added a quick function to handle that

In [26]:
def left_quat_matrix(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    
                            #w
                            #x
                            #y
                            #z
    return sym.Matrix([
        [qw, -qx, -qy, -qz], 
        [qx,  qw, -qz,  qy], 
        [qy,  qz,  qw, -qx], 
        [qz, -qy,  qx,  qw]  
    ])

def quat_inv(q):
    return sym.Matrix([q[0], -q[1], -q[2], -q[3]])
    
def quat_norm(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    return q / sym.sqrt(q[0]**2 + q[1]**2 + q[2]**2 + q[3]**2)

# State transition function

## Orientation

$$
\begin{align}
f_q &: \mathbb{R}^4 \times \mathbb{R}^3 \times \mathbb{R} \rightarrow \mathbb{R}^4 \\
f_q(q_k, u_{g, k}, \Delta t_k) &= \text{normalize} \left( q_k \otimes \begin{bmatrix} 1 \\ \frac{1}{2} u_{g, k} \Delta t_k \end{bmatrix} \right) \\
\text{normalize}(x) &= \frac{x}{\|x\|}
\end{align}
$$

In [40]:
gyro_rotation = 0.5 * u_g * dt
gyro_rotation_as_quat = sym.Matrix.vstack(sym.Matrix([1]), gyro_rotation)

f_q_unnormed = left_quat_matrix(x_q) * gyro_rotation_as_quat

# normalization is only needed in the case of actual hardware implementation.
# Since floating point inaccuracy can influence the results,
# this would insure that the orientation quaternion stays normalized.
# In the case of the theoretical implementation, I don't need to normalize since I assume infinite precision
# f_q = quat_norm(f_q_unnormed)

f_q = f_q_unnormed
f_q

⎡-0.5⋅\Delta t⋅gₓ⋅qₓ - 0.5⋅\Delta t⋅g_y⋅q_y - 0.5⋅\Delta t⋅g_z⋅q_z + q_w⎤
⎢                                                                       ⎥
⎢0.5⋅\Delta t⋅gₓ⋅q_w - 0.5⋅\Delta t⋅g_y⋅q_z + 0.5⋅\Delta t⋅g_z⋅q_y + qₓ ⎥
⎢                                                                       ⎥
⎢0.5⋅\Delta t⋅gₓ⋅q_z + 0.5⋅\Delta t⋅g_y⋅q_w - 0.5⋅\Delta t⋅g_z⋅qₓ + q_y ⎥
⎢                                                                       ⎥
⎣-0.5⋅\Delta t⋅gₓ⋅q_y + 0.5⋅\Delta t⋅g_y⋅qₓ + 0.5⋅\Delta t⋅g_z⋅q_w + q_z⎦

## Position & Velocity

### Acceleration first

$$
f_{a}(q_k,u_{a, k} ) = \left( q_k \otimes \begin{bmatrix} 0 \\ u_{a,k} \end{bmatrix} \otimes q_k^{-1} \right)_{1:3} - \begin{bmatrix} 0 \\ 0 \\ 9.81 \end{bmatrix}
$$

In [28]:
a_rot1 = left_quat_matrix(x_q) * sym.Matrix([0, a_x, a_y, a_z])
a_rot2 = left_quat_matrix(a_rot1) * quat_inv(x_q)

pure_accel = sym.simplify(a_rot2)[1:4, 0]
f_a = pure_accel - sym.Matrix([0, 0, gravity])

f_a

⎡       q_w⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y) + qₓ⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z) + q_y⋅(-aₓ⋅q_y + a_y⋅qₓ + a_
⎢                                                                                                     
⎢       q_w⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ) - qₓ⋅(-aₓ⋅q_y + a_y⋅qₓ + a_z⋅q_w) + q_y⋅(aₓ⋅qₓ + a_y⋅q_y + a_z
⎢                                                                                                     
⎣-g_{gravity} + q_w⋅(-aₓ⋅q_y + a_y⋅qₓ + a_z⋅q_w) + qₓ⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ) - q_y⋅(aₓ⋅q_w - a_y⋅

z⋅q_w) - q_z⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ)        ⎤
                                                ⎥
⋅q_z) + q_z⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y)        ⎥
                                                ⎥
q_z + a_z⋅q_y) + q_z⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z)⎦

### Integrate acceleration second

$$
\begin{align}
f_{vp} &: \mathbb{R}^4 \times \mathbb{R}^3 \times \mathbb{R}^3 \times \mathbb{R}^3 \times \mathbb{R} \rightarrow \mathbb{R}^6 \\
f_{vp} (q_k, v_k, p_k, u_{a, k}, \Delta t_k) &= \begin{bmatrix} 
v_k + f_a \cdot \Delta t \\
p_k + (v_k + f_a \cdot \Delta t) \cdot \Delta t
\end{bmatrix}
\end{align}
$$

In [29]:
f_v = x_v + f_a * dt
f_p = x_p + (x_v + f_a * dt) * dt

f_v, f_p

⎛⎡        \Delta t⋅(q_w⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y) + qₓ⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z) + q_y⋅(-aₓ⋅q_y +
⎜⎢                                                                                                    
⎜⎢       \Delta t⋅(q_w⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ) - qₓ⋅(-aₓ⋅q_y + a_y⋅qₓ + a_z⋅q_w) + q_y⋅(aₓ⋅qₓ + a_
⎜⎢                                                                                                    
⎝⎣\Delta t⋅(-g_{gravity} + q_w⋅(-aₓ⋅q_y + a_y⋅qₓ + a_z⋅q_w) + qₓ⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ) - q_y⋅(aₓ

 a_y⋅qₓ + a_z⋅q_w) - q_z⋅(aₓ⋅q_z + a_y⋅q_w - a_z⋅qₓ)) + vₓ        ⎤  ⎡        \Delta t⋅(\Delta t⋅(q_w⋅
                                                                  ⎥  ⎢                                
y⋅q_y + a_z⋅q_z) + q_z⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y)) + v_y        ⎥, ⎢       \Delta t⋅(\Delta t⋅(q_w⋅(
                                                                  ⎥  ⎢                                
⋅q_w - a_y⋅q_z + a_z⋅q_y) + q_z⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z)) + v_z⎦  ⎣\D

## Orientation, Velovity & Position combined

$$
\begin{align}
x_{k + 1} &= f(x_k, u_k, \Delta t_k) \\
&= f\left(\begin{bmatrix} f_q(q_k, u_{g, k}, \Delta t_k) \\ f_{vp} (q_k, v_k, p_k, u_{a, k}, \Delta t_k) \end{bmatrix}\right) \\
\end{align}
$$

In [41]:
f_state_transition = sym.Matrix.vstack(f_q, f_v, f_p)
f_state_transition

⎡                                                        -0.5⋅\Delta t⋅gₓ⋅qₓ - 0.5⋅\Delta t⋅g_y⋅q_y - 
⎢                                                                                                     
⎢                                                        0.5⋅\Delta t⋅gₓ⋅q_w - 0.5⋅\Delta t⋅g_y⋅q_z + 
⎢                                                                                                     
⎢                                                        0.5⋅\Delta t⋅gₓ⋅q_z + 0.5⋅\Delta t⋅g_y⋅q_w - 
⎢                                                                                                     
⎢                                                        -0.5⋅\Delta t⋅gₓ⋅q_y + 0.5⋅\Delta t⋅g_y⋅qₓ + 
⎢                                                                                                     
⎢                \Delta t⋅(q_w⋅(aₓ⋅q_w - a_y⋅q_z + a_z⋅q_y) + qₓ⋅(aₓ⋅qₓ + a_y⋅q_y + a_z⋅q_z) + q_y⋅(-a
⎢                                                                        

# Measurement function

$$
h(x_k) = \left( q_k^{-1} \otimes \begin{bmatrix} 0 \\ 0 \\ 0 \\ g \end{bmatrix} \otimes q_k \right)_{1:3}
$$

In [31]:
h_rot1 = left_quat_matrix(quat_inv(x_q)) * sym.Matrix([0, 0, 0, gravity])
h_rot2 = left_quat_matrix(h_rot1) * x_q

h_measurement = h_rot2[1:4, 0]
h_measurement

⎡             -2⋅g_{gravity}⋅q_w⋅q_y + 2⋅g_{gravity}⋅qₓ⋅q_z              ⎤
⎢                                                                        ⎥
⎢              2⋅g_{gravity}⋅q_w⋅qₓ + 2⋅g_{gravity}⋅q_y⋅q_z              ⎥
⎢                                                                        ⎥
⎢               2                 2                  2                  2⎥
⎣g_{gravity}⋅q_w  - g_{gravity}⋅qₓ  - g_{gravity}⋅q_y  + g_{gravity}⋅q_z ⎦

# Jacobian Matrices

In [32]:
A = f_state_transition.jacobian(x)
A

⎡                     1                                     -0.5⋅\Delta t⋅gₓ                          
⎢                                                                                                     
⎢              0.5⋅\Delta t⋅gₓ                                      1                                 
⎢                                                                                                     
⎢              0.5⋅\Delta t⋅g_y                             -0.5⋅\Delta t⋅g_z                         
⎢                                                                                                     
⎢              0.5⋅\Delta t⋅g_z                             0.5⋅\Delta t⋅g_y                          
⎢                                                                                                     
⎢\Delta t⋅(2⋅aₓ⋅q_w - 2⋅a_y⋅q_z + 2⋅a_z⋅q_y)   \Delta t⋅(2⋅aₓ⋅qₓ + 2⋅a_y⋅q_y + 2⋅a_z⋅q_z)    \Delta t⋅
⎢                                                                        

In [24]:
H = h_measurement.jacobian(x)
H

⎡-2⋅g_{gravity}⋅q_y  2⋅g_{gravity}⋅q_z  -2⋅g_{gravity}⋅q_w  2⋅g_{gravity}⋅qₓ   0  0  0  0  0  0⎤
⎢                                                                                              ⎥
⎢ 2⋅g_{gravity}⋅qₓ   2⋅g_{gravity}⋅q_w  2⋅g_{gravity}⋅q_z   2⋅g_{gravity}⋅q_y  0  0  0  0  0  0⎥
⎢                                                                                              ⎥
⎣2⋅g_{gravity}⋅q_w   -2⋅g_{gravity}⋅qₓ  -2⋅g_{gravity}⋅q_y  2⋅g_{gravity}⋅q_z  0  0  0  0  0  0⎦

# Lambdify functions and dill

In [65]:
f_q_input_symbols = list(x_q) + list(u_g) + [dt]
f_a_input_symbols = list(x_q) + list(u_a) + [gravity]

# If I forget to add values to the lambdify functions,
# the function will later on assume that the ones I forgot are global variables.
# If those global variables ar not set then, an error will be raised.
f_q_missing_symbols = f_q.free_symbols - set(f_q_input_symbols)
if (f_q_missing_symbols):
    raise ValueError(f"CRITICAL: Forgot to include these symbols in lambdify: {f_q_missing_symbols}")
f_a_missing_symbols = f_a.free_symbols - set(f_a_input_symbols)
if missing_symbols:
    raise ValueError(f"CRITICAL: Forgot to include these symbols in lambdify: {f_a_missing_symbols}")

f_q_lambda = sym.lambdify(f_q_input_symbols, f_q, 'numpy')
f_a_lambda = sym.lambdify(f_a_input_symbols, f_a, 'numpy')

f_a_lambda

<function _lambdifygenerated(q_w, q_x, q_y, q_z, a_x, a_y, a_z, g_gravity)>

In [70]:
A_input_symbols = list(x) + list(u) + [dt, gravity]
H_input_symbols = list(q) + [gravity]

A_missing_symbols = A.free_symbols - set(A_input_symbols)
if (A_missing_symbols):
    raise ValueError(f"CRITICAL: Forgot to include these symbols in lambdify: {A_missing_symbols}")
H_missing_symbols = H.free_symbols - set(H_input_symbols)
if (H_missing_symbols):
    raise ValueError(f"CRITICAL: Forgot to include these symbols in lambdify: {H_missing_symbols}")

A_lambda = sym.lambdify(A_input_symbols, A, 'numpy')
H_lambda = sym.lambdify(H_input_symbols, H, 'numpy')
A_lambda

<function _lambdifygenerated(q_w, q_x, q_y, q_z, v_x, v_y, v_z, p_x, p_y, p_z, a_x, a_y, a_z, g_x, g_y, g_z, _Dummy_57, g_gravity)>

At the very end, rerun the whole code to export the lambdified functions and matrices.

In [72]:
with open("observer_functions_and_matrices.pkl", "wb") as f:
    dill.dump({
        "f_q": f_q_lambda,
        "f_a": f_a_lambda,
        "A": A_lambda,
        "H": H_lambda
    }, f)
    
# import in python script with
#
# with open("observer_functions_and_matrices.pkl", "rb") as f:
#     ekf_funcs = dill.load(f)
#
# A = ekf_funcs["A"]